# ⚠️ SUPERSEDED (PLAN.md §34 N.6)

`protein_selector.pipeline.run_pipeline` was retired when the orchestration layer became
`pipelines/` + `stages/` + `nodes/`. Use the named pipeline instead:

```python
from protein_selector.pipelines.course_candidates_pipeline import CourseCandidatesConfig, run
run(CourseCandidatesConfig(), db_path=Path('cache/protein_selector.db'))
```

or, from a shell, `make ray-plan` / `make ray-run`. The cells below are kept as a record
of the old invocation and will not execute.


# Run the real pipeline

Unlike `build_demo_db.ipynb` (hand-authored, illustrative data, no network
calls), this notebook calls `protein_selector.pipeline.run_pipeline` for
real: real RCSB hard-filters search, real simulability/composition checks,
real ligand CCD/SMILES lookup, real RDKit (and Meeko, if installed)
parameterizability, real Europe PMC literature counts, and a real AlphaFold
DB lookup for the modeling exercise -- every number below comes from
an actual API response, nothing is invented.

**Stage config is grouped into dataclasses** (`CandidateSearchConfig`, `CandidateFilterConfig`, `MdSimulationConfig`,
`PocketDetectionConfig`, ...) -- see `pipeline.py`'s own docstring for the full
list and what each corresponds to -- each validator module's own `EXERCISE_NAME`
constant (`modeling`/`md_simulation`/`docking`) is the single source of
truth for that stage's label, inherited by every downstream column/table (PLAN.md §7b).

MD simulation and pocket detection need the conda-only
`environment-validation.yml` env (a local `fpocket` binary too, for pocket
detection) -- this notebook enables both below, so it must be run through that
conda env's Python (the "Python 3 (protein-selector-validation, conda)" kernel),
not the base venv. Docking (real Vina + PLIP) still isn't wired into the
pipeline at all -- no receptor-prep/pocket-center-extraction code exists yet to
auto-run it, so it will always show `"not_run"` regardless of config.

`max_candidates` is kept small (a real Meeko 3D-embedding step, if
installed, can be slow for some real bound ligands) -- raise it once you've
confirmed a run completes in reasonable time for your machine.

Requires the `notebook` dependency group: `uv sync --group notebook`.

In [7]:
import logging
from pathlib import Path

import pandas as pd

from protein_selector.pipeline import (
    CandidateFilterConfig,
    CandidateSearchConfig,
    MdSimulationConfig,
    PocketDetectionConfig,
    run_pipeline,
)

logging.basicConfig(level=logging.INFO, format="%(message)s")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Separate db from build_demo_db.ipynb's illustrative one, so the two never mix.
DB_PATH = Path("cache/protein_selector_real.db")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
CSV_PATH = Path("report_real.csv")

In [ ]:
rows = run_pipeline(
    db_path=DB_PATH,
    candidate_search=CandidateSearchConfig(max_candidates=10000, random_seed=3000),
    # candidates bigger than 50 residues are dropped from the pool entirely --
    # see CandidateFilterConfig's own docstring for why residues, not atoms.
    candidate_filter=CandidateFilterConfig(max_residues=150),
    # modeling_lookup left at its default (ModelingLookupConfig(enabled=True))
    # -- the real AlphaFold DB fetch-only check, persisted as exercise "modeling".
    md_simulation=MdSimulationConfig(
        enabled=True,
        n_steps=50,
        max_minimization_iterations=10,
    ),
    pocket_detection=PocketDetectionConfig(enabled=True),
    report_csv_path=CSV_PATH,
)
len(rows)

🔎 search: pool_size 200000 exceeds RCSB's 10000-row ceiling, clamping
HTTP Request: POST https://search.rcsb.org/rcsbsearch/v2/query "HTTP/1.1 200 OK"
HTTP Request: POST https://search.rcsb.org/rcsbsearch/v2/query "HTTP/1.1 200 OK"
🔎 search: sampled 10000 of 10000 matching candidates
HTTP Request: POST https://data.rcsb.org/graphql "HTTP/1.1 200 OK"
HTTP Request: POST https://data.rcsb.org/graphql "HTTP/1.1 200 OK"
HTTP Request: POST https://data.rcsb.org/graphql "HTTP/1.1 200 OK"
HTTP Request: POST https://data.rcsb.org/graphql "HTTP/1.1 200 OK"
HTTP Request: POST https://data.rcsb.org/graphql "HTTP/1.1 200 OK"
HTTP Request: POST https://data.rcsb.org/graphql "HTTP/1.1 200 OK"
HTTP Request: POST https://data.rcsb.org/graphql "HTTP/1.1 200 OK"
HTTP Request: POST https://data.rcsb.org/graphql "HTTP/1.1 200 OK"
HTTP Request: POST https://data.rcsb.org/graphql "HTTP/1.1 200 OK"
HTTP Request: POST https://data.rcsb.org/graphql "HTTP/1.1 200 OK"
HTTP Request: POST https://data.rcsb.org/grap

## The real joined report

In [ ]:
from protein_selector.core.report import rows_to_dataframe

report_df = rows_to_dataframe(rows)
report_df

,pdb_id,uniprot_id,title,organism,n_residues,n_atoms,resolution,method,n_protein_entities,ligand_ccd,ligand_smiles,ligand_rdkit_parameterizable,ligand_meeko_parameterizable,pocket_druggable,pocket_druggability_score,completeness,nonstd_residues,literature_count,suitable_for,modeling_alphafold_entry_id,modeling_alphafold_mean_plddt,modeling_alphafold_low_confidence_fraction,md_simulation_pdbfixer_repaired,modeling_status,modeling_predicted_difficulty,modeling_measured_difficulty,modeling_gap,modeling_tier,modeling_failure_mode,modeling_notes,md_simulation_status,md_simulation_predicted_difficulty,md_simulation_measured_difficulty,md_simulation_gap,md_simulation_tier,md_simulation_failure_mode,md_simulation_notes,docking_status,docking_predicted_difficulty,docking_measured_difficulty,docking_gap,docking_tier,docking_failure_mode,docking_notes,rationale_json
0,101M,P02185,SPERM WHALE MYOGLOBIN F46V N-BUTYL ISOCYANIDE ...,Physeter macrocephalus,154,1413,2.07,X-RAY DIFFRACTION,1,HEM,None,True,False,None,NaN,1.000000,False,0.0,"[""modeling""]",AF-P02185-F1,97.50,0.000,None,pass,0.000,0.031135,0.031135,intro,NaN,"[""AF-P02185-F1: mean pLDDT 97.5, 0.0% low/very...",not_run,0.447111,NaN,NaN,core,NaN,[],not_run,1.0000,None,None,challenge,None,[],"{""simulability_reasons"": [], ""pocket_reasons"":..."
1,102L,P00720,HOW AMINO-ACID INSERTIONS ARE ALLOWED IN AN AL...,Tequatrovirus T4,165,1439,1.74,X-RAY DIFFRACTION,1,BME,None,True,True,None,NaN,0.987879,False,5.0,[],NaN,NaN,NaN,None,fail,NaN,1.000000,NaN,challenge,completeness,"[""no AlphaFold DB entry found for UniProt acce...",not_run,0.419374,NaN,NaN,core,NaN,[],not_run,0.0000,None,None,intro,None,[],"{""simulability_reasons"": [], ""pocket_reasons"":..."
2,102M,P02185,SPERM WHALE MYOGLOBIN H64A AQUOMET AT PH 9.0,Physeter macrocephalus,154,1423,1.84,X-RAY DIFFRACTION,1,HEM,None,True,False,None,NaN,1.000000,False,0.0,"[""modeling""]",AF-P02185-F1,97.50,0.000,None,pass,0.000,0.032788,0.032788,intro,NaN,"[""AF-P02185-F1: mean pLDDT 97.5, 0.0% low/very...",not_run,0.416444,NaN,NaN,core,NaN,[],not_run,1.0000,None,None,challenge,None,[],"{""simulability_reasons"": [""residue count 154 o..."
3,10PA,Q2G0L4,Crystal structure of SdrD A2-A3 domains from S...,Staphylococcus aureus subsp. aureus JH1,321,2574,1.92,X-RAY DIFFRACTION,1,CA,None,True,True,None,NaN,0.987539,False,0.0,[],AF-Q2G0L4-F1,71.69,0.358,None,fail,0.358,1.000000,0.642000,challenge,confidence,"[""35.8% of AF-Q2G0L4-F1 is low/very-low pLDDT,...",not_run,0.593487,NaN,NaN,core,NaN,[],not_run,0.0000,None,None,intro,None,[],"{""simulability_reasons"": [""residue count 321 o..."
4,107M,P02185,SPERM WHALE MYOGLOBIN V68F N-BUTYL ISOCYANIDE ...,Physeter macrocephalus,154,1418,2.09,X-RAY DIFFRACTION,1,HEM,None,True,False,None,NaN,1.000000,False,0.0,"[""modeling""]",AF-P02185-F1,97.50,0.000,None,pass,0.000,0.038397,0.038397,intro,NaN,"[""AF-P02185-F1: mean pLDDT 97.5, 0.0% low/very...",not_run,0.449778,NaN,NaN,core,NaN,[],not_run,1.0000,None,None,challenge,None,[],"{""simulability_reasons"": [""residue count 154 o..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
165,1LDS,P61769,Crystal Structure of monomeric human beta-2-mi...,Homo sapiens,100,913,1.80,X-RAY DIFFRACTION,1,NA,None,True,False,False,0.368,0.970000,False,47.0,"[""modeling"", ""md_simulation""]",AF-P61769-F1,94.06,0.059,True,pass,0.059,0.034812,-0.024188,intro,NaN,"[""AF-P61769-F1: mean pLDDT 94.1, 5.9% low/very...",pass,0.361111,0.013107,-0.348004,intro,NaN,"[""1640 atoms, 50 steps at 2.0 fs completed cle...",not_run,0.8160,None,None,challenge,None,[],"{""simulability_reasons"": [], ""pocket_reasons"":..."
166,1CAA,P24297,X-RAY CRYSTAL STRUCTURES OF THE OXIDIZED AND R...,Pyrococcus furiosus,53,475,1.80,X-RAY DIFFRACTION,1,FE,None,True,True,False,0.358,1.000000,False,8.0,"[""modeling"", ""md_simulation""]",AF-P24297-F1,97.12,0.019,True,pass,0.019

## What's real here, spelled out

- `title`/`organism`/`n_residues`/`resolution`/... -- real RCSB entry metadata.
- `ligand_rdkit_parameterizable` -- a real RDKit sanitization result (and real Meeko
  3D-embed + PDBQT-write result in `ligand_meeko_parameterizable`, if the `validate`
  extra is installed).
- `literature_count` -- a real Europe PMC hit count for this exact PDB ID.
- `modeling_status`/`modeling_predicted_difficulty` -- a real AlphaFold
  DB lookup: if the candidate's UniProt accession has a modeled entry, this reflects its
  actual published confidence fractions; if not, `modeling_status` is `"fail"`
  with `FailureMode.COMPLETENESS`, not guessed.

In [ ]:
report_df[["pdb_id", "uniprot_id", "ligand_ccd", "ligand_rdkit_parameterizable", "literature_count", "modeling_status", "modeling_predicted_difficulty", "modeling_failure_mode"]]

,pdb_id,uniprot_id,ligand_ccd,ligand_rdkit_parameterizable,literature_count,modeling_status,modeling_predicted_difficulty,modeling_failure_mode
0,101M,P02185,HEM,True,0.0,pass,0.000,NaN
1,102L,P00720,BME,True,5.0,fail,NaN,completeness
2,102M,P02185,HEM,True,0.0,pass,0.000,NaN
3,10PA,Q2G0L4,CA,True,0.0,fail,0.358,confidence
4,107M,P02185,HEM,True,0.0,pass,0.000,NaN
...,...,...,...,...,...,...,...,...
165,1LDS,P61769,NA,True,47.0,pass,0.059,NaN
166,1CAA,P24297,FE,True,8.0,pass,0.019,NaN
167,1KTH,P12111,PO4,True,13.0,pass,0.117,NaN
168,1GV5,P06876,NA,True,2.0,fail,0.683,confidence
